In [156]:
import pandas as pd
from PIL import Image
import io
import requests
from IPython.display import Image, display, clear_output
import ipywidgets as widgets
import json
import time

In [129]:
# load key from json 
with open('keys/omdb_key.json', 'r') as f:
    keys = json.load(f)
api_key = keys["omdb_key"]

In [130]:
# store address
ombapi_url = "http://www.omdbapi.com/"

# use variablae from json, no real key in the code
params = {
    "i": "tt3890160", # lookup by imbd id
    "apikey": api_key
}

# send the request to OMDB and wait for the reply
# actual call to the API: go fetch from this 
# response: stores everything OMDb sends back in a variable
response = requests.get(ombapi_url, params=params)
# check if the request works
print(response.status_code) 

200


In [131]:
# converts texts into a python dictionary and stored in "data"
data = response.json()
print(data)

{'Title': 'Baby Driver', 'Year': '2017', 'Rated': 'R', 'Released': '28 Jun 2017', 'Runtime': '113 min', 'Genre': 'Action, Crime, Drama', 'Director': 'Edgar Wright', 'Writer': 'Edgar Wright', 'Actors': 'Ansel Elgort, Jon Bernthal, Jon Hamm', 'Plot': 'Coerced into working for a crime boss, a young getaway driver must face the music when a doomed heist threatens his life, love, and freedom.', 'Language': 'English, American Sign Language', 'Country': 'United Kingdom, United States', 'Awards': 'Nominated for 3 Oscars. 44 wins & 67 nominations total', 'Poster': 'https://m.media-amazon.com/images/M/MV5BMjM3MjQ1MzkxNl5BMl5BanBnXkFtZTgwODk1ODgyMjI@._V1_QL75_UX380_CR0,0,380,562_.jpg', 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '7.5/10'}, {'Source': 'Rotten Tomatoes', 'Value': '92%'}, {'Source': 'Metacritic', 'Value': '86/100'}], 'Metascore': '86', 'imdbRating': '7.5', 'imdbVotes': '696,803', 'imdbID': 'tt3890160', 'Type': 'movie', 'DVD': 'N/A', 'BoxOffice': '$107,825,862', 'Produ

In [157]:
print(data["Poster"])
display(Image(url=data["Poster"], width=150))

https://m.media-amazon.com/images/M/MV5BMjM3MjQ1MzkxNl5BMl5BanBnXkFtZTgwODk1ODgyMjI@._V1_QL75_UX380_CR0,0,380,562_.jpg


In [159]:
def get_poster(imdb_id):
    params = {"i": imdb_id, "apikey": api_key}
    response = requests.get("http://www.omdbapi.com/", params=params)
    data = response.json()
    if data.get("Response") == "False":
        return None

In [160]:
def get_poster(imdb_id):
    params = {"i": imdb_id, "apikey": api_key}
    try:
        response = requests.get(ombapi_url, params=params, timeout=10)
        data = response.json()
    except Exception as e:
        print("error on", imdb_id, e)
        return None

    if data.get("Response") == "False":
        print("failed:", imdb_id, data.get("Error"))
        return None
    poster = data.get("Poster")
    if poster == "N/A":
        return None
    return poster

print(get_poster("tt3890160"))   # test: should print a url

https://m.media-amazon.com/images/M/MV5BMjM3MjQ1MzkxNl5BMl5BanBnXkFtZTgwODk1ODgyMjI@._V1_QL75_UX380_CR0,0,380,562_.jpg


In [136]:
test_ids = df["imdbID"].head(5)

for imdb_id in test_ids:
    print(imdb_id, get_poster(imdb_id))
    time.sleep(0.2)

tt0068646 https://m.media-amazon.com/images/M/MV5BNGEwYjgwOGQtYjg5ZS00Njc1LTk2ZGEtM2QwZWQ2NjdhZTE5XkEyXkFqcGc@._V1_QL75_UY562_CR8,0,380,562_.jpg
tt0047478 https://m.media-amazon.com/images/M/MV5BZjliMWExOTMtZDQ3ZS00NWU3LWIyN2EtMjllNzk3ZTNlYzg4XkEyXkFqcGc@._V1_QL75_UY562_CR9,0,380,562_.jpg
tt0050083 https://m.media-amazon.com/images/M/MV5BYjE4NzdmOTYtYjc5Yi00YzBiLWEzNDEtNTgxZGQ2MWVkN2NiXkEyXkFqcGc@._V1_QL75_UX380_CR0,11,380,562_.jpg
tt0108052 https://m.media-amazon.com/images/M/MV5BNjM1ZDQxYWUtMzQyZS00MTE1LWJmZGYtNGUyNTdlYjM3ZmVmXkEyXkFqcGc@._V1_QL75_UX380_CR0,4,380,562_.jpg
tt0071562 https://m.media-amazon.com/images/M/MV5BMDIxMzBlZDktZjMxNy00ZGI4LTgxNDEtYWRlNzRjMjJmOGQ1XkEyXkFqcGc@._V1_QL75_UX380_CR0,4,380,562_.jpg


In [168]:
posters = []
top200 = df.iloc[100:200]

for i, imdb_id in enumerate(top200["imdbID"]):
    posters.append(get_poster(imdb_id))
    if i % 25 == 0:
        print(f"{i} / {len(top200)} done")
    time.sleep(0.2)

print("finished")

0 / 100 done
25 / 100 done
50 / 100 done
75 / 100 done
finished


In [170]:
movies = pd.read_csv("data/movies_with_posters.csv")

missing = movies[movies["Poster"].isna()]
print(len(missing), "posters still needed")
for count, (i, row) in enumerate(missing.iterrows()):
    movies.loc[i, "Poster"] = get_poster(row["imdbID"])
    if count % 25 == 0:
        print(f"{count} / {len(missing)} done")
        movies.to_csv("data/movies_with_posters.csv", index=False)
    time.sleep(0.2)

movies.to_csv("data/movies_with_posters.csv", index=False)
print(movies["Poster"].notna().sum(), "/", len(movies), "posters")

349 posters still needed
0 / 349 done
25 / 349 done
50 / 349 done
75 / 349 done
100 / 349 done
125 / 349 done
150 / 349 done
175 / 349 done
200 / 349 done
225 / 349 done
250 / 349 done
275 / 349 done
300 / 349 done
325 / 349 done
449 / 449 posters


In [138]:
poster_df = pd.DataFrame({"imdbID": top100["imdbID"], "Poster": posters})

movies = df.merge(poster_df, on="imdbID", how="left")

print(movies.shape)                     # still (449, 27)
print(movies["Poster"].isna().sum(), "movies without a poster")
movies[["Title", "Poster"]].head()

(449, 27)
349 movies without a poster


,Title,Poster
0,The Godfather,https://m.media-amazon.com/images/M/MV5BNGEwYj...
1,Seven Samurai,https://m.media-amazon.com/images/M/MV5BZjliMW...
2,12 Angry Men,https://m.media-amazon.com/images/M/MV5BYjE4Nz...
3,Schindler's List,https://m.media-amazon.com/images/M/MV5BNjM1ZD...
4,The Godfather: Part II,https://m.media-amazon.com/images/M/MV5BMDIxMz...


In [139]:
movies.to_csv("data/movies_with_posters.csv", index=False)

In [104]:
movies = pd.read_csv("data/movies_with_posters.csv")
movies["Decade"] = movies["Year"] // 10 * 10

In [105]:
genres = sorted(movies["Genre"].str.split(", ").explode().unique())
decades = sorted(movies["Decade"].unique())

print(genres)
print(decades)

['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'Thriller', 'War', 'Western']
[np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960), np.int64(1970), np.int64(1980), np.int64(1990), np.int64(2000), np.int64(2010), np.int64(2020)]


In [106]:
genre_dd = widgets.Dropdown(options=["Any"] + genres, description="Genre:")
decade_dd = widgets.Dropdown(options=[("Any", None)] + [(f"{d}s", d) for d in decades], description="Era:")
rating_slider = widgets.FloatSlider(value=8.0, min=movies["IMDb_10"].min(), max=movies["IMDb_10"].max(), step=0.1, description="Min IMDb:")

recommend_btn = widgets.Button(description="🎬 Recommend", button_style="info")
reveal_btn = widgets.Button(description="Reveal", button_style="success", disabled=True)
out = widgets.Output()

current_picks = None
guess_boxes = []

In [161]:
def recommend(b):
    global current_picks, guess_boxes

    picks = movies
    if genre_dd.value != "Any":
        picks = picks[picks["Genre"].str.contains(genre_dd.value, regex=False)]
    if decade_dd.value is not None:
        picks = picks[picks["Decade"] == decade_dd.value]
    picks = picks[picks["IMDb_10"] >= rating_slider.value]

    with out:
        clear_output()
        if len(picks) < 3:
            print(f"only {len(picks)} movies match — loosen your filters")
            reveal_btn.disabled = True
            return

        current_picks = picks.sample(3)
        guess_boxes = []
        cards = []

        for _, m in current_picks.iterrows():
            poster = widgets.HTML(f'<img src="{m["Poster"]}" width="150">') if pd.notna(m["Poster"]) else widgets.HTML("no poster")
            title = widgets.HTML(f"<b>{m['Title']}</b> ({m['Year']})")
            guess = widgets.FloatText(value=8.0, step=0.1, layout=widgets.Layout(width="150px"))
            guess_boxes.append(guess)
            cards.append(widgets.VBox([poster, title, guess]))

        display(widgets.HBox(cards))
        print("guess each movie's IMDb rating (out of 10), then hit Reveal")
        reveal_btn.disabled = False


def reveal(b):
    total = 0
    with out:
        print("\n--- results ---")
        for (_, m), box in zip(current_picks.iterrows(), guess_boxes):
            real = m["IMDb_10"]
            diff = abs(box.value - real)
            pts = 3 if diff <= 0.3 else 1 if diff <= 0.7 else 0
            total += pts
            print(f"{m['Title']}: real {real} | you guessed {box.value} | +{pts}")
        print(f"\ntotal: {total} / 9")
    reveal_btn.disabled = True

In [162]:
recommend_btn.on_click(recommend)
reveal_btn.on_click(reveal)

display(widgets.VBox([
    genre_dd, decade_dd, rating_slider,
    widgets.HBox([recommend_btn, reveal_btn]),
    out
]))

In [151]:
df = pd.read_csv('data/movies_with_posters.csv')
df.head()

,Rank,Title,Year,Genre,Director,Cast,Language,Plot,Awards,Production,...,Audience_Rating,Audience_Reviews,Letterboxd,Letterboxd_Votes,Google_Score,Streaming_On,RT_URL,imdbID,Custom_Score,Poster
0,1,The Godfather,1972,"Drama, Crime",Francis Ford Coppola,"Marlon Brando, Al Pacino, James Caan, Robert D...",en,"Spanning the years 1945 to 1955, a chronicle o...",Won 3 Oscars. Another 30 wins & 19 nominations.,Paramount Pictures,...,95.0,1087289.0,90,1789555.0,90.0,"paramount, itunes, amzvid",https://www.rottentomatoes.com/m/godfather,tt0068646,93.83,https://m.media-amazon.com/images/M/MV5BNGEwYj...
1,2,Seven Samurai,1954,"Action, Drama",Akira Kurosawa,"Toshirō Mifune, Takashi Shimura, Yoshio Inaba,...",ja,A samurai answers a village's request for prot...,Nominated for 2 Oscars. Another 6 wins & 5 nom...,Columbia Pictures,...,94.0,167123.0,92,318537.0,93.0,"hbo, itunes, amzvid, hbm",https://www.rottentomatoes.com/m/seven_samurai...,tt0047478,93.33,https://m.media-amazon.com/images/M/MV5BZjliMW...
2,3,12 Angry Men,1957,Drama,Sidney Lumet,"Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",en,The defense and the prosecution have rested an...,Nominated for 3 Oscars. Another 16 wins & 8 no...,Criterion Collection,...,94.0,376697.0,92,917590.0,91.0,"itunes, amzvid",https://www.rottentomatoes.com/m/1000013_12_an...,tt0050083,92.83,https://m.media-amazon.com/images/M/MV5BYjE4Nz...
3,4,Schindler's List,1993,"Drama, History, War",Steven Spielberg,"Liam Neeson, Ben Kingsley, Ralph Fiennes, Caro...",en,The true story of how businessman Oskar Schind...,Won 7 Oscars. Another 74 wins & 26 nominations.,Universal Pictures,...,94.0,601555.0,90,978054.0,93.0,"itunes, amzvid",https://www.rottentomatoes.com/m/schindlers_list,tt0108052,92.50,https://m.media-amazon.com/images/M/MV5BNjM1ZD...
4,5,The Godfather: Part II,1974,"Drama, Crime",Francis Ford Coppola,"Al Pacino, Robert Duvall, Diane Keaton, Robert...",en,In the continuing saga of the Corleone crime f...,Won 6 Oscars. Another 13 wins & 16 nominations.,Paramount Pictures,...,94.0,586600.0,92,934120.0,89.0,"paramount, itunes, amzvid",https://www.rottentomatoes.com/m/godfather_par...,tt0071562,91.67,https://m.media-amazon.com/images/M/MV5BMDIxMz...


In [152]:
TITLE = "Title"
SCORE = "IMDb_10"
POSTER = "Poster"

In [163]:
titles = sorted(df[TITLE].dropna().astype(str).unique())

p1 = widgets.Combobox(placeholder="type a movie", options=titles, description="Player 1:")
p2 = widgets.Combobox(placeholder="type a movie", options=titles, description="Player 2:")
btn = widgets.Button(description="Battle!", button_style="danger")
out = widgets.Output()

def find(name):
    match = df[df[TITLE].astype(str).str.lower() == name.strip().lower()]
    return None if match.empty else match.iloc[0]

def show(movie):
    print(f"{movie[TITLE]}  |  {SCORE}: {movie[SCORE]}")
    if POSTER and pd.notna(movie[POSTER]):
        display(Image(url=movie[POSTER], width=150))

def battle(b):
    with out:
        clear_output()
        m1, m2 = find(p1.value), find(p2.value)
        if m1 is None or m2 is None:
            print("movie not found, check the spelling")
            return
        show(m1)
        print("   VS")
        show(m2)
        if m1[SCORE] > m2[SCORE]:
            print(f"\n🏆 {m1[TITLE]} wins!")
        elif m2[SCORE] > m1[SCORE]:
            print(f"\n🏆 {m2[TITLE]} wins!")
        else:
            print("\nit's a tie!")

btn.on_click(battle)
display(p1, p2, btn, out)

Combobox(value='', description='Player 1:', options=('12 Angry Men', '12 Years a Slave', '13th', '1917', '2001…

Combobox(value='', description='Player 2:', options=('12 Angry Men', '12 Years a Slave', '13th', '1917', '2001…

Button(button_style='danger', description='Battle!', style=ButtonStyle())

Output()